# Mathematical Basics

This notebook covers the material in the CWJSN Mathematical Basics chapters. Since much of this material is also covered in the Math Foundations repo, I will only add interesting new stuff I learn along the way.

In [2]:
# Set up Revise.jl for automatic code reloading
# This only needs to be run once at the beginning of your notebook session
using Revise

# Load the package 
using Calculus
using Latexify

## From Calculator to Computer Chapter

### Big Numbers

Two interesting points I picked up in this chapter.
1. In julia you can express a large number using underscores for better readability, e.g., `1_000_000` instead of $1000000$
2. You need to be careful when you run into big integers. So $10^{20}$ gives a wrong number with no error or warning, while the same value typed as a literal comes out right. If a computation might pass about $9.2 \times 10^{18}$ you need to use `big(...)`

In [3]:
# literals: Julia widens a literal that does not fit in Int64
v1 = 10_000_000_000_000_000_000
v2 = 1_000_000_000_000_000_000_000_000_000_000_000_000_000
# arithmetic: no widening
v3 = 10^20
v4 = big(10)^20
v5 = big"123_456"

L"""
\begin{array}{lrll}
\textbf{Julia} & \textbf{value} & \textbf{type} & \\ \hline
\mathtt{10\_000\_000\_000\_000\_000\_000} & %$v1 & \mathtt{%$(typeof(v1))} & \text{literal, widened} \\
\mathtt{1\_000\_000\_000\_000\_000\_000\_000\_000\_000\_000\_000\_000} & %$v2 & \mathtt{%$(typeof(v2))} & \text{literal, widened} \\ \hline
10^{20} & %$v3 & \mathtt{%$(typeof(v3))} & \text{wrong: overflow wraps around} \\
\mathrm{big}(10)^{20} & %$v4 & \mathtt{%$(typeof(v4))} & \text{correct} \\
\mathtt{big}\text{"}\mathtt{123\_456}\text{"} & %$v5 & \mathtt{%$(typeof(v5))} & \text{string form takes underscores too}
\end{array}
"""


"\$\\begin{array}{lrll}\n\\textbf{Julia} & \\textbf{value} & \\textbf{type} & \\\\ \\hline\n\\mathtt{10\\_000\\_000\\_000\\_000\\_000\\_000} & 10000000000000000000 & \\mathtt{Int128} & \\text{literal, widened} \\\\\n\\mathtt{1\\_000\\_000\\_000\\_000\\_000\\_000\\_000\\_000\\_000\\_000\\_000\\_000} & 10000000" ⋯ 114 bytes ⋯ "20 & \\mathtt{Int64} & \\text{wrong: overflow wraps around} \\\\\n\\mathrm{big}(10)^{20} & 100000000000000000000 & \\mathtt{BigInt} & \\text{correct} \\\\\n\\mathtt{big}\\text{\"}\\mathtt{123\\_456}\\text{\"} & 123456 & \\mathtt{BigInt} & \\text{string form takes underscores too}\n\\end{array}\n\$"

Note: I am keeping an ugly version of formatting here just to show how latexify does stuff with display:

This is how you do it wrong:

In [4]:
display(@latexdefine a = 10^20)

L"$a = 10^{20} = 7766279631452241920$"

This is how you do it correctly:

In [5]:
display(@latexdefine b = big(10)^20)
display(@latexdefine c = 10^big(20))

L"$b = \left( \mathrm{big}\left( 10 \right) \right)^{20} = 100000000000000000000$"

L"$c = 10^{\mathrm{big}\left( 20 \right)} = 100000000000000000000$"

And here is an alternate way to format all of the above a lot nicer:

In [6]:
a = 10^20; b = big(10)^20; c = 10^big(20)

L"""
\begin{aligned}
&\text{Wrong:}  && 10^{20} = %$a \\
&\text{Right:}  && \mathrm{big}(10)^{20} = %$b \\
&               && 10^{\mathrm{big}(20)} = %$c
\end{aligned}
"""


L"$\begin{aligned}
&\text{Wrong:}  && 10^{20} = 7766279631452241920 \\
&\text{Right:}  && \mathrm{big}(10)^{20} = 100000000000000000000 \\
&               && 10^{\mathrm{big}(20)} = 100000000000000000000
\end{aligned}
$"

In the CWJSN text, when discussing big integers, Professor Verzani states:

>The user is expected to have a sense that they need to be careful when their values are quite large. But the better recommendation is that the user use floating point numbers, which as easy as typing `2.0^63`. Though not always exact, floating point values can represent a much bigger range values and are exact for a reasonably wide range of integer values.

Is this a good recommendation?

If we are just interested in sheer size of numbers, then it is true that floating point values reach much farther than Int64:


| type | largest value | exact for every integer up to |
|---|---|---|
| `Int64` | $2^{63}-1 \approx 9.2 \times 10^{18}$ | all of them (then wraps silently) |
| `Float64` | $\approx 1.8 \times 10^{308}$ | $2^{53} \approx 9.0 \times 10^{15}$ (then rounds silently) |
| `BigInt` | limited only by memory | all of them |

However, as Verzani himself notes, these larger values are not always exact. In fact, `Float64`'s exact range is smaller than `Int64`'s. Past $2^{53}$, floats skip integers: `2.0^53 + 1 == 2.0^53` is true. So switching to floats trades one silent failure (wrap-around) for another (rounding), and the rounding starts about a thousand times sooner:


| Julia | exact? |
|---|---|
| `2.0^63` | yes — a power of two |
| `10.0^20` | yes |
| `10.0^23` | no — stored as $99999999999999991611392$ |
| `prod(1.0:21)` | yes, $= 21!$ (its factors of 2 keep it representable) |
| `prod(1.0:25)` | no — off from $25!$ by $71303168$ |

So there should be two recommendations, not one:

* For exact integers, use `big`, which will always give exact results.
* For magnitude only, use floats, and accept about 16 significant digits.

Floats are the better recommendation only when you don't care about the last digits.

We can see this played out in the Homer Simpson example. The board in the photo claims $3987^{12} + 4365^{12} = 4472^{12}$. The text instead writes the
expression

$$
3987^{12} + 4365^{12} - 4472^{12}
$$

because the equation holds **exactly when that difference is $0$** — so "is it $0$?" is the test. The claim is the answer is yes on calculators, but not in Julia!

| Julia | result | note |
|---|---|---|
| `3987^12 + 4365^12 - 4472^12` | `-842986283280935454` | `Int64` overflow: meaningless |
| `3987.0^12 + 4365.0^12 - 4472.0^12` | `1.2118937808588159e33` | floats: right size, last digits off |
| `big(3987)^12 + big(4365)^12 - big(4472)^12` | `1211886809373872630985912112862690` | exact, and not $0$: no counterexample |

### Factorial

The text mentions `factorial` but doesnt show you how to use it. Below is a table that does.

In [7]:
# Run an expression; return its value, or the error it threw instead of stopping the cell.
r1 = attempt(() -> factorial(5))
r2 = attempt(() -> factorial(20))
r3 = attempt(() -> factorial(21))
r4 = attempt(() -> factorial(big(21)))
r5 = attempt(() -> prod(1:21))

L"""
\begin{array}{lrl}
\textbf{Julia} & \textbf{result} & \textbf{type} \\ \hline
\mathtt{factorial(5)}      & %$r1 & \mathtt{%$(typeof(r1))} \\
\mathtt{factorial(20)}     & %$r2 & \mathtt{%$(typeof(r2))} \\
\mathtt{factorial(21)}     & \text{error} & \mathtt{%$(typeof(r3))} \\
\mathtt{factorial(big(21))} & %$r4 & \mathtt{%$(typeof(r4))} \\
\mathtt{prod(1{:}21)}      & %$r5 & \mathtt{%$(typeof(r5))}
\end{array}
"""


L"$\begin{array}{lrl}
\textbf{Julia} & \textbf{result} & \textbf{type} \\ \hline
\mathtt{factorial(5)}      & 120 & \mathtt{Int64} \\
\mathtt{factorial(20)}     & 2432902008176640000 & \mathtt{Int64} \\
\mathtt{factorial(21)}     & \text{error} & \mathtt{OverflowError} \\
\mathtt{factorial(big(21))} & 51090942171709440000 & \mathtt{BigInt} \\
\mathtt{prod(1{:}21)}      & -4249290049419214848 & \mathtt{Int64}
\end{array}
$"

In [8]:
L"""
\begin{array}{l}
\text{Notice how unlike the power function, the factorial function in Julia will throw an error} \\
\text{if the result does not fit in the default integer type. In the example shown, we get an} \\
\text{error message which actually suggests a solution!:} \\[1ex]
%$(latex_error(r3))
\end{array}
"""


L"$\begin{array}{l}
\text{Notice how unlike the power function, the factorial function in Julia will throw an error} \\
\text{if the result does not fit in the default integer type. In the example shown, we get an} \\
\text{error message which actually suggests a solution!:} \\[1ex]
\quad \text{OverflowError: 21 is too large to look up in the table}\text{;} \\ \quad \text{consider using }\mathtt{factorial(big(21))}\text{ instead}
\end{array}
$"

Also notice that the `prod` function multiplies in `Int64` and wraps around silently like `^`. If you want, use `prod(big(1):big(21))`. 

Note that `factorial` only works for integers. For non-integers use the `gamma` function, which extends it to real (and complex) numbers using an integral. Since that is related to future lessons, we will skip the details here. The important thing to remember at this point, is that `gamma` shifts by 1 relative to factorial.

You can see examples of all of the above in the table below:

In [9]:
@variables n
q1 = attempt(() -> prod(big(1):big(21)))
q2 = attempt(() -> factorial(5.0))
q3 = attempt(() -> factorial(-1))
q4 = attempt(() -> factorial(n))
q5 = attempt(() -> gamma(6))
q6 = attempt(() -> gamma(4.5))
q7 = attempt(() -> gamma(big(31)) == factorial(big(30)))

L"""
\begin{array}{lrll}
\textbf{Julia} & \textbf{result} & \textbf{type} & \textbf{note} \\ \hline
\mathtt{prod(big(1){:}big(21))}   & %$(latex_cell(q1)) & %$(latex_type(q1)) & = 21! \text{, no wrap-around} \\ \hline
\mathtt{factorial(5.0)}           & %$(latex_cell(q2)) & %$(latex_type(q2)) & \text{integers only} \\
\mathtt{factorial(-1)}            & %$(latex_cell(q3)) & %$(latex_type(q3)) & \text{not defined for negatives} \\
\mathtt{factorial(n)}             & %$(latex_cell(q4)) & %$(latex_type(q4)) & \text{no symbolic factorial} \\ \hline
\mathtt{gamma(6)}                 & %$(latex_cell(q5)) & %$(latex_type(q5)) & \Gamma(6) = 5! \\
\mathtt{gamma(4.5)}               & %$(latex_cell(q6)) & %$(latex_type(q6)) & \Gamma(4.5) = 3.5! \\
\mathtt{gamma(big(31)) == factorial(big(30))} & %$(latex_cell(q7)) & %$(latex_type(q7)) & \Gamma(n+1) = n!
\end{array}
"""

"\$\\begin{array}{lrll}\n\\textbf{Julia} & \\textbf{result} & \\textbf{type} & \\textbf{note} \\\\ \\hline\n\\mathtt{prod(big(1){:}big(21))}   & 51090942171709440000 & \\mathtt{BigInt} & = 21! \\text{, no wrap-around} \\\\ \\hline\n\\mathtt{factorial(5.0)}           & \\text{error} & \\mathtt{M" ⋯ 278 bytes ⋯ "6)}                 & 120.0 & \\mathtt{Float64} & \\Gamma(6) = 5! \\\\\n\\mathtt{gamma(4.5)}               & 11.631728396567448 & \\mathtt{Float64} & \\Gamma(4.5) = 3.5! \\\\\n\\mathtt{gamma(big(31)) == factorial(big(30))} & \\mathtt{true} & \\mathtt{Bool} & \\Gamma(n+1) = n!\n\\end{array}\n\$"

### Numeric Literals

In the CWJSN section on `Numeric literals` the author asks what is the value of `2pi^2pi` and the author notes that the correct answer is `2 * (pi ^ (2pi))` because this is the "usual mathematical convention". He does not make clear why that is the case. The explanation is: the juxtaposed product `2pi` (a numeric literal `2`, written directly against the named `pi`) to the right of the exponent is read as a single term - i.e. the exponent itself. By contrast, using PEMDAS the power symbol (`^`) is tied to the term immediately to the left, i.e. the `pi`. Once the exponential term is calculated, we multiply it by the leftmost term. 

Be sure to note that in this example, PEMDAS applies only to the left hand side while the first rule applies only to juxtaposition. Without the juxtaposition, PEMDAS always applies. In the case of an explicit `*` with no juxtaposition, the usual PEMDAS order returns, so `2^3*x` is $2^3 \cdot x$, not $2^{3x}$ (last row in table below). Hence the author's advice to be explicit.

The table below gives examples of how Julia applies its rules, and shows why by first translating the Julia code to standard mathematical notation:

In [10]:
let x = 2
    p1, o1 = 2pi^2pi, 2 * (pi^2) * pi
    p2, o2 = 2x^3,    (2x)^3
    p3, o3 = 2^3x,    2^3 * x
    p4, o4 = 2^3*x,   2^(3x)

L"""
\begin{array}{llrlr}
\textbf{typed} & \textbf{Julia reads} & \textbf{value} & \textbf{not} & \textbf{value} \\ \hline
\texttt{2pi\textasciicircum{}2pi} & 2\pi^{2\pi} & %$(round(p1, digits = 2)) & 2\pi^{2}\cdot\pi & %$(round(o1, digits = 2)) \\
\texttt{2x\textasciicircum{}3}    & 2x^{3}      & %$p2 & (2x)^{3}  & %$o2 \\
\texttt{2\textasciicircum{}3x}    & 2^{3x}      & %$p3 & 2^{3}\cdot x & %$o3 \\ \hline
\texttt{2\textasciicircum{}3*x}   & 2^{3}\cdot x & %$p4 & 2^{3x} & %$o4
\end{array}
\qquad (x = %$x)
"""
end

L"$\begin{array}{llrlr}
\textbf{typed} & \textbf{Julia reads} & \textbf{value} & \textbf{not} & \textbf{value} \\ \hline
\texttt{2pi\textasciicircum{}2pi} & 2\pi^{2\pi} & 2658.98 & 2\pi^{2}\cdot\pi & 62.01 \\
\texttt{2x\textasciicircum{}3}    & 2x^{3}      & 16 & (2x)^{3}  & 64 \\
\texttt{2\textasciicircum{}3x}    & 2^{3x}      & 64 & 2^{3}\cdot x & 16 \\ \hline
\texttt{2\textasciicircum{}3*x}   & 2^{3}\cdot x & 16 & 2^{3x} & 64
\end{array}
\qquad (x = 2)
$"

### Logarithms in Julia vs on paper

| Julia | result | meaning |
|---|---|---|
| `log(x)` | `log(ℯ)` = `1` | natural log, $\ln x$ |
| `log10(x)` | `log10(1000)` = `3.0` | base 10, $\log_{10} x$ |
| `log2(x)` | `log2(8)` = `3.0` | base 2, $\log_2 x$ |
| `log(b, x)` | `log(2, 8)` = `3.0` | base $b$, $\log_b x = \dfrac{\ln x}{\ln b}$ |
| `ln(x)` | not defined | Julia has no `ln` |

On paper: school/engineering write $\log x$ for base 10 and $\ln x$ for base $e$; higher mathematics
(and Julia) write $\log x$ for the natural log, because

$$\frac{d}{dx}\ln x = \frac{1}{x} \qquad\text{while}\qquad \frac{d}{dx}\log_{10} x = \frac{1}{x \ln 10}.$$

Trap: `log(10, 1000)` = `2.9999999999999996` (it computes $\ln 1000 / \ln 10$); use `log10`.


---

### CWJSN calculator: Questions

Values marked * were not in my REPL session; they were added afterwards by Claude to complete the comparison.

| Q | Question (CWJSN) | My Julia → result | Answer | Why |
|---|---|---|---|---|
| 1 | Compute $22/7$ | `22/7` → `3.142857142857143` | `3.142857142857143` | `/` always returns a `Float64` |
| 2 | Compute $\sqrt{220}$ | `sqrt(220)` → `14.832396974191326` | `14.832396974191326` | `sqrt` always returns a `Float64`, even for a perfect square: `sqrt(16)` → `4.0`\* |
| 3 | Compute $2^8$ | `2^8` → `256` | `256` | integer base and exponent stay `Int` |
| 4 | Compute $\frac{9 - 5 \cdot (3-4)}{6 - 2}$ | `(9 - 5* (3-4))/(6-2)` → `3.5` | `3.5` | numerator and denominator each need a pair of parentheses |
| 5 | Compute $\frac{(.25 - .2)^2}{(1/4)^2 + (1/3)^2}$ | `(.25 - .2)^2/((1/4)^2 + (1/3)^2)` → `0.014399999999999993` | `0.014399999999999993` | exact value is $0.0144$; `0.25 - 0.2` → `0.04999999999999999`\* because $0.2$ has no exact binary form |
| 6 | Decimal value of $1 + \frac{1}{2} + \frac{1}{2^2} + \frac{1}{2^3} + \frac{1}{2^4}$ | `1 + 1/2 + 1/2^2 + 1/2^3 + 1/2^4` → `1.9375` | `1.9375` | `^` before `/`, so `1/2^2` is $1/4$; exact, since powers of 2 are exact in binary |
| 7 | Compute $\frac{3 - 2^2}{4 - 2\cdot3}$ | `(3-2^2)/(4-(2*3))` → `0.5` | `0.5` | the inner `(2*3)` is optional: `*` before `-` |
| 8 | Compute $(1/2) \cdot 32 \cdot 3^2 + 100 \cdot 3 - 20$ | `(1/2) * 32 * 3^2 + 100 * 3 -20` → `424.0` | `424.0` | a `Float64` because of `1/2` |
| 9a | Oven at 200 Celsius: closest Fahrenheit of 350, 375, 400, 425, 450? | `((9/5) * 200) + 32` → `392.0` | 400 | $F = 9/5 \cdot C + 32$ |
| 9b | Internal 165 Fahrenheit: closest Celsius of 68, 70, …, 80? | `(5/9) * (165-32)` → `73.88888888888889` | 74 | $C = 5/9\,(F - 32)$ |
| 10 | Least parentheses for $\frac{3 - 2}{4 - 1}$ | `(3-2)/(4-1)` → `0.3333333333333333`; `(3 - 2)/ 4 - 1` → `-0.75`\*; `3 - 2 / (4 - 1)` → `2.3333333333333335`\* | `(3 - 2) / (4 - 1)` | `/` binds tighter than `-`, so both pairs are needed |
| 11 | Least parentheses for $\frac{3\cdot2}{4}$ | `3*2 /4` → `1.5`; `(3*2) /4` → `1.5` | `3 * 2 / 4` | `*` and `/` have equal precedence and go left to right, so `3*2` is done first anyway |
| 12 | Least parentheses for $2^{4 - 2}$ | `2 ^ (4-2)` → `4`; `2 ^ 4 - 2` → `14`\*; `(2 ^ 4) - 2` → `14`\* | `2 ^ (4 - 2)` | `^` binds tighter than `-`, so the exponent needs its pair |
| 13 | The Office: `9653 +` gives `11532`; what was added? | `11532- 9653` → `1879` | `1879` | check: `9653 + 1879` → `11532`\* |
| 14 | A more familiar expression for `1 / 2 / 3 / 4 / 5 / 6` | `1 / 2 / 3 / 4 / 5 / 6` → `0.001388888888888889`; `1 / (2 / 3 / 4 / 5 / 6)` → `180.0`; `1 /(2 * 3 * 4 * 5 * 6)` → `0.001388888888888889`; `1 / 2 * 3 / 4  * 5 / 6` → `0.3125`; `1 / 2 / 3 / 4 / 5 / 6 *  (2/2) * (3/3) * (4/4) * (5/5) * (6/6)` → `0.001388888888888889` | `1 /(2 * 3 * 4 * 5 * 6)` | `/` goes left to right, so dividing by each number in turn is dividing by their product. `2 / 3 / 4 / 5 / 6` is $2/360 = 1/180$, so its reciprocal is 180; `1 / 2 * 3 / 4 * 5 / 6` is $\frac{1\cdot3\cdot5}{2\cdot4\cdot6} = \frac{15}{48}$ |
| 15 | Which gives a different answer: `2 - 3 - 4`, `(2 - 3) - 4`, `2 - (3 - 4)` | `(2 - 3) - 4` → `-5`; `2 - (3 - 4)` → `3`; `2 - 3 - 4` → `-5` | `2 - (3 - 4)` | `-` is left-associative: `2 - 3 - 4` means `(2 - 3) - 4` |
| 16 | Which gives a different answer: `2 - 3 * 4`, `(2 - 3) * 4`, `2 - (3 * 4)` | `(2 - 3) * 4` → `-4`; `2 - (3 * 4)` → `-10`; `2 - 3 * 4` → `-10` | `(2 - 3) * 4` | `*` before `-`, so `2 - 3 * 4` means `2 - (3 * 4)` |
| 17 | Which gives a different answer: `-1^2`, `(-1)^2`, `-(1^2)` | `-(1^2)` → `-1`; `-1^2` → `-1`; `(-1)^2` → `1` | `(-1)^2` | `^` before unary minus: `-1^2` parses as `-(1 ^ 2)`\* |
| 18 | Value of $\sin(\pi/10)$ | `sin(pi/10)` → `0.3090169943749474` | `0.3090169943749474` | `sin` takes radians |
| 19 | Value of $\sin(52^\circ)$ | `sind(52)` → `0.788010753606722` | `0.788010753606722` | `sind` takes degrees |
| 20 | Value of $\frac{\sin(\pi/3) - 1/2}{\pi/3 - \pi/6}$ | `(sin(pi/3)-(1/2))/((pi/3)-(pi/6))` → `0.6990570277140041` | `0.6990570277140041` | the whole denominator needs one pair; the inner pairs are optional |
| 21 | Is $\sin^{-1}(\sin(3\pi/2))$ equal to $3\pi/2$? | `asin(sin(3pi/2))` → `-1.5707963267948966`; `3pi/2` → `4.71238898038469` | No | `asin` returns an angle in $\lbrack -\pi/2, \pi/2 \rbrack$; $\sin(3\pi/2) = -1$, and the angle in that range with sine $-1$ is $-\pi/2$ (`asin(sin(3pi/2)) == -pi/2` → `true`\*) |
| 22 | Value of `round(3.5000)` | `round(3.5000)` → `4.0` | `4.0` | `round` keeps the type (`Float64`); `round(Int, 3.5)` → `4`\*. Ties go to the even neighbour: `round(2.5)` → `2.0`\* |
| 23 | Value of `sqrt(32 - 12)` | `sqrt(32 - 12)` → `4.47213595499958` | `4.47213595499958` | $\sqrt{20} = 2\sqrt{5}$; `2sqrt(5)` → `4.47213595499958`\* |
| 24 | Which is greater, $e^\pi$ or $\pi^e$? | `ℯ^pi > pi^ℯ` → `true` (`ℯ` typed as `\euler` then Tab) | $e^\pi$ | `ℯ^pi` → `23.140692632779267`\*, `pi^ℯ` → `22.459157718361038`\*. Without the Tab: `exp(pi) > pi^exp(1)` → `true`\* (`exp(1)` is $e$ as a `Float64`), or `MathConstants.e`, the same constant as `ℯ`\*; `using Base.MathConstants: e` then lets you write plain `e`\*. A plain `e` is not predefined |
| 25 | Value of $\pi - (x - \sin(x)/\cos(x))$ when $x = 3$ | `x = 3 ; pi - (x - sin(x)/cos(x))` → `-0.0009538894844847157` | `-0.0009538894844847157` | `x - sin(x)/cos(x)` → `3.142546543074278`\*: one Newton step for $\sin x = 0$ from $x = 3$, already close to $\pi$ |
| 26 | What is $10!$? | `factorial(10)` → `3628800` | `3628800` | `!` is not a factorial operator in Julia |
| 27 | Will `-2^2` produce `4` or `-4`? | `-2^2` → `-4` | `-4` | as Q17: parses as `-(2 ^ 2)`\* |
| 28a | Twitter post: $8 \div 2(2+2)$ — what is the answer by strict left-to-right PEMDAS? | `8/2*(2+2)` → `16.0`; `8/(2*(2+2))` → `1.0` | `16` | the chapter takes strict left-to-right PEMDAS, treating `2(2+2)` as an ordinary multiplication; reading implied multiplication as binding tighter gives 1. A choice of convention |
| 28b | Does `Julia` give the strict left-to-right PEMDAS answer for `8÷2(2+2)`? Why or why not? | `8÷2(2+2)` → `1`; `8 ÷ (2 * (2 + 2))` → `1` | No; Julia gives numeric literal coefficients higher precedence than `*`, `/` and `÷` | Julia reads `8 ÷ (2 * (2 + 2))`; `÷` is integer division, so the result is an `Int` |
| 29 | Sagan's number, 10 sextillion ($10^{22}$): which entry does *not* work? `10*10^21`, `10*10.0^21`, `10e21`, `1e22`, `10_000_000_000_000_000_000_000` | `10*10^21` → `1864712049423024128`\*; `10*10.0^21` → `1.0e22`\*; `10e21` → `1.0e22`\*; `1e22` → `1.0e22`\*; `10_000_000_000_000_000_000_000` → `10000000000000000000000`\* | `10*10^21` | `10^21` has an integer base, so it overflows `Int64` and wraps silently: `10^18` → `1000000000000000000`\* is the last power that fits; `10^19` → `-8446744073709551616`\*, past `typemax(Int64)` = `9223372036854775807`\*. A float base (`10.0^21`) or a float literal (`10e21`, `1e22`) does not overflow; the long literal is parsed as `Int128`\* |
| 30 | Planck length $l_p = \sqrt{\hbar G / c^3}$ comes out near $10^{-32}$, not $10^{-35}$. Why? | with `c = 299_792_458;` `planck_length = sqrt(h_bar * G / c^3)` → `3.5793584153084007e-32`; with `c = big(299_792_458)` → `1.6162550244237053…e-35` (full value below) | "`Julia` computes this value incorrectly" | `c^3` overflows `Int64` silently: `299_792_458^3` → `5493783091195596520`\*, but the true value is `26944002417373989539335912`\*, above `typemax(Int64)` = `9223372036854775807`\*. `big(c)` fixes it, as does a float: `c = 299792458.0` gives `1.6162550244237053e-35`\* |

Q30 with `c = big(299_792_458)`, in full: `1.616255024423705337061393098475417498117496219791154915828327386777545429675462e-35`


## Variables Chapter

### Variable gotchas

The text notes the danger of shadowing which can overwrite the meaning of a name binding. Specifically let's look at the example of `x + y = 3`, which is a function definition in Julia's short _assignment form_ of functions. Running it redefines the name binding for `+` and detaches it from the `Base.:+` function, and attaches it to the newly defined function. Let's see how this plays out.

In [11]:
let
    println("Inside a 'let' block, we define the new function x + y = 3 and this is what happens:")
    x + y = 3
    @show 1 + 5
    @show Base.:+(1, 5)
end
println("Outside the 'let' block, the original definition of '+' is unchanged.")
@show 1 + 5
println("In this part of the notebook (and similarly in the CLI, we are in 'Main' module).")
println("So here is what happens when we define a simple function in the 'Main' module.")
x + y = 3
@show 1 + 5
println("After redefining '+' again using 'const (+) = Base.:+', the following results are observed:")
const (+) = Base.:+
@show 1 + 5
println("It is important to note that this is just restoring the 'Base' definition of '+' in the 'Main' module.")
println("However it is not actually deleting the new definition of '+'. It is still there, but nothing refers to it.")
println("If you re-run this cell, the new definition will add a method, +(::Any, ::Any), to Base's own + function")
println("rather than defining a new function.")
println("On first run the line \"a\" + \"b\" creates an error. On the second run this line gives an answer")
try
    @show "a" + "b"
catch err
    println(typeof(err))
end
println("Note: the simplest way to delete the new method completely is to restart the kernel.")

Inside a 'let' block, we define the new function x + y = 3 and this is what happens:
1 + 5 = 3
Base.:+(1, 5) = 6
Outside the 'let' block, the original definition of '+' is unchanged.
1 + 5 = 6
In this part of the notebook (and similarly in the CLI, we are in 'Main' module).
So here is what happens when we define a simple function in the 'Main' module.
1 + 5 = 3
After redefining '+' again using 'const (+) = Base.:+', the following results are observed:
1 + 5 = 6
It is important to note that this is just restoring the 'Base' definition of '+' in the 'Main' module.
However it is not actually deleting the new definition of '+'. It is still there, but nothing refers to it.
If you re-run this cell, the new definition will add a method, +(::Any, ::Any), to Base's own + function
rather than defining a new function.
On first run the line "a" + "b" creates an error. On the second run this line gives an answer
MethodError
Note: the simplest way to delete the new method completely is to restart th

In [13]:
println("After you restart the kernel and run the cell above only once, try this")
println("Add the function and then use `Base.delete_method` to remove it")
x + y = 3
Base.delete_method(which(+, (String, String)))
try
    @show "a" + "b"
catch err
    println(first(split(sprint(showerror, err), "\n")))
end
println("After using `Base.delete_method` to delete the method, trying to add two strings results in an error.")


After you restart the kernel and run the cell above only once, try this
Add the function and then use `Base.delete_method` to remove it
MethodError: no method matching +(::String, ::String)
After using `Base.delete_method` to delete the method, trying to add two strings results in an error.


### Euler's number: use `ℯ`, not `e`

**NB:** This is actually a gotcha in _Calculus with Julia Squared_ (inherited from _Calculus with Julia_), not a specifically Julia gotcha.

Julia's Euler's number is `ℯ`, typed `\euler` then Tab. Like `π`, it is an `Irrational`: it has no fixed precision, and it turns into a number of the right type only when it is combined directly with a number (e.g. `2ℯ`, `ℯ + 1`). This is equally true if this expression is evaluated inside a `Symbolics.jl` expression. The exception where `ℯ` is never turned into a number, but is still meaningfully evaluated, is where Julia has a special method (e.g. `ℯ^2` is computed as `exp(2)`).

By contrast, bare `e` is not defined in Julia (`Base.MathConstants.e` is the same `ℯ`). `using Calculus` brings in CalculusWithJuliaSquared's `e`, which is `const e = exp(1)`, an ordinary `Float64` rounded to about 16 digits. So `e === ℯ` is `false`.

Write `b` for either one (measured with Julia 1.13.2, CalculusWithJuliaSquared 0.17.0, Symbolics 7.45.0, `@variables x`):

| computation | `b = e` | `b = ℯ` |
|---|---|---|
| `typeof(b)` | `Float64` | `Irrational{:ℯ}` |
| `b^2 - exp(2)` | `-8.881784197001252e-16` | `0.0` |
| `log(b)` | `1.0` | `1` |
| `big(b) - big(ℯ)` | `-1.4456…e-16` | `0.0` |
| `b^x` | `2.718281828459045^x` | `exp(x)` |
| `Symbolics.derivative(b^x, x)` | a float power times a float log (below) | `exp(x)` |
| `b*x` | `2.718281828459045x` | `ℯ*x` |
| `2b*x` | `5.43656365691809x` | `5.43656365691809x` |
| `2*(b*x)` | `5.43656365691809x` | `2ℯ*x` |

With `e` the derivative comes back as `(2.718281828459045^x)*log(2.718281828459045)`. That is numerically $e^x$, but Symbolics only sees an arbitrary float base, so the simplification $\frac{d}{dx}e^x = e^x$ never appears.

Why `ℯ` gives exact results: Julia defines `ℯ^y` as `exp(y)` and `log(ℯ)` as the integer `1` (special methods in `Base.MathConstants`). `big(ℯ)` is correct to the full `BigFloat` precision, while `big(e)` converts the 16-digit float and is wrong after that.

The _CalculusWithJuliaSquared_ text (`precalc/exp_log_functions`) inherits Verzani's comment that the float `e` "loses the feature of arbitrary precision, but has other advantages" and does not say which. The obvious one is that `e` can be typed on any keyboard. In ordinary floating-point arithmetic the two mostly agree, because `ℯ` becomes a `Float64` as soon as it is used in an expression with a number (`2ℯ` is `5.43656365691809`). For anything exact or symbolic, use `ℯ` (or `exp(...)`).

---

### CWJSN variables: Questions

Values marked * were not in my REPL session; they were added afterwards by Claude to complete the comparison.

| Q | Question (CWJSN) | My Julia → result | Answer | Why |
|---|---|---|---|---|
| 1 | Let $a=10$, $b=2.3$, $c=8$. Find $(a-b)/(a-c)$ | `a, b, c = 10, 2.3, 8`; `(a − b)/(a − c)` → `3.85` | `3.85` | $7.7/2$. The `−` here is the Unicode minus sign (U+2212), copied from the page's math, not the keyboard `-`; Julia's parser turns it into the ordinary `-`, so `a − b` is `a - b` |
| 2 | Let `x = 4`. Compute $y=100 - 2x - x^2$ | `x = 4`; `y = 100 - 2x -x^2` → `76` | `76` | $100 - 8 - 16$ |
| 3 | Value of `a^b - b^a` after `a = 3.2; b=2.3` | `a = 3.2; b=2.3` → `2.3`; `a^b - b^a` → `0.14354012963861962` | `0.14354012963861962` | `;` puts two statements on one line; only the last value is shown |
| 4 | $\frac{p - q}{\sqrt{p(1-p)}}$ for $p=0.25$, $q=0.2$, computed in parts | `p, q = 0.25, 0.2`; `top = p - q` → `0.04999999999999999`; `bottom = sqrt(p*(1-p))` → `0.4330127018922193`; `ans = top/bottom` → `0.11547005383792514` | `0.11547005383792514` | `top` is not exactly $0.05$ because $0.2$ has no exact binary form (as in the calculator Q5) |
| 5 | $y = \frac{x^2 - 2x - 8}{x^2 - 9x - 20}$ for $x=3$, with variables for the top and the bottom | `x = 3`; `top = x^2 - 2x - 8` → `-5`; `bottom = x^2 - 9x -20` → `-38`; `y = top/bottom` → `0.13157894736842105` | `0.13157894736842105` | $\frac{-5}{-38} = \frac{5}{38}$ |
| 6 | Which is not a valid variable name: `5degreesbelowzero`, `some_really_long_name…` (full name below), `aMiXeDcAsEnAmE`, `fahrenheit451` | `5degreesbelowzero = 1` → `ERROR: syntax: "5" is not a valid function argument name`; `some_really_long_name… = 1` → `1`\*; `aMiXeDcAsEnAmE = 1` → `1`\*; `fahrenheit451 = 1` → `1`\* | `5degreesbelowzero` | a name cannot start with a digit. Julia reads `5degreesbelowzero` as the numeric-literal product `5 * degreesbelowzero`, so `= 1` is a short-form definition of `*` with `5` as an argument name, the same pattern as `x + y = 3` above |
| 7 | Which is one of Julia's built-in math constants: `pi`, `oo`, `E`, `I` | `pi` → `π = 3.1415926535897...`; `oo` → `UndefVarError`\*; `E` → `UndefVarError`\*; `I` → `UndefVarError`\* in bare Julia, `UniformScaling{Bool}(true)`\* after `using Calculus` | `pi` | `oo`, `E` and `I` are SymPy's names for $\infty$, $e$ and $i$ (Julia's are `Inf`, `ℯ` and `im`). The `I` that `using Calculus` brings in is LinearAlgebra's identity matrix, not a constant |
| 8 | Which key sequence produces `δ = 1/10`: `\delta[tab] = 1/10`, `delta[tab] = 1/10`, `$\delta$ = 1/10` | `δ = 1/10` → `0.1`; `delta = 1/10` → `0.1`\*; `$\delta$ = 1/10` → `ERROR: syntax: "$" expression outside quote`\* | `\delta[tab] = 1/10` | without the backslash, Tab does not convert, so `delta` stays an ordinary name; `$` is interpolation, valid only inside strings and quoted code |
| 9 | Which is not a valid way to assign three variables at once: `a=1, b=2, c=3`, `a,b,c = 1,2,3`, `a=1; b=2; c=3` | `a,b,c = 1,2,3` → `(1, 2, 3)`; `a=1, b=2, c=3` → `ERROR: syntax: invalid assignment location "2"`; `a=1; b=2; c=3` → `3` | `a=1, b=2, c=3` | the commas build tuples, so it parses as `a = ((1, b) = ((2, c) = 3))`, and `2` cannot be assigned to |
| 10 | `x = y = z = 3` will always: assign all three variables to `3`, create 3 linked values that stay synced, or throw an error | `x = y = z = 3` → `3`; `x, y, z` → `(3, 3, 3)`; `x = 4`; `(x, y, z)` → `(4, 3, 3)`\* | assign all three variables at once to `3` | `=` associates right to left and returns its right-hand side: `x = (y = (z = 3))`. The names are not linked: rebinding `x` leaves `y` and `z` alone |

Q6, the full name: `some_really_long_name_that_is_no_fun_to_type`

## Number System Chapter

### Some Gotchas

#### `1.`, `2.*3` and the broadcasting dot

In this chapter, in the "Heads up" paragraph regarding `2.*3`, Verzani brings up broadcasting without explaining it. Broadcasting is a powerful operation in Julia which applies a function or operator to each element of a collection such as a vector or tuple. The table below shows how broadcasting works, and why the `.` causes parsing ambiguity.

Measured with Julia 1.13.2:

| input | result |
|---|---|
| `1.` | `1.0` (a `Float64`) |
| `2. * 3` | `6.0`: the dot belongs to the `2` |
| `2 .* 3` | `6`: the dot belongs to the `*` (broadcast) |
| `2 .* (3, 6)` | `(6, 12)`: the dot belongs to the `*` (a more realistic broadcast EXAMPLE) |
| `2.*3` | `ParseError: ambiguous '.' syntax; add whitespace to clarify` |


#### Number Representation

Verzani mentions the problem of rounding error, where for example `sqrt(2) * sqrt(2) - 2` produces the very small number `4.440892098500626e-16` instead of zero. What he leaves off is the standard  solution used to address this problem in Julia: `sqrt(2) * sqrt(2) ≈ 2` which yields `true`. 

He also seems to imply that integer overflow is in a different category than rounding errors. The source of the error is actually the same - the size of memory used to store numbers. How the error manifests itself is slightly different: a seemingly arbitrary result as opposed to a tiny residue. But the residue is also not mathematically correct and does not exist according to mathematical definitions. Rather, it arises only inside a computer because a number with no finite binary expansion (such as $\sqrt{2}$ or $1/10$) cannot be stored exactly in a finite number of bits.

Hence, the key to avoiding and handling these errors is understanding how numbers are represented in computers. That is why in the earlier section I stressed that taking the power of a float instead of an integer is not always the best solution, as you are just substituting one memory issue for another. What does differ is how to fix the problem:

- **Overflow** is purely a size limit, and more bits remove it completely: `big(2)^64` yields `18446744073709551616`, exact.
- **Rounding** is also about the binary format: $\sqrt{2}$ and $1/10$ have no finite binary expansion, so more bits shrink the error but never remove it. `sqrt(big(2)) * sqrt(big(2)) - 2` yields `-1.7272…e-77`, still not zero. Hence use the function `isapprox()` or `≈` when comparing floats.
- Float overflow also looks different from integer overflow: `2.0^1024` yields `Inf`, whereas `typemax(Int64) + 1` wraps to `-9223372036854775808`.
  
In the next cell you can see examples (created by Claude) of what is discussed here.

#### Overflow, rounding and `≈`

Measured with Julia 1.13.2.

| error | example → result | why | fix → result |
|---|---|---|---|
| integer overflow | `typemax(Int64) + 1` → `-9223372036854775808`; `2^64` → `0` | `Int64` arithmetic wraps around silently | more bits remove it: `big(2)^64` → `18446744073709551616`, exact |
| float rounding | `sqrt(2) * sqrt(2) - 2` → `4.440892098500626e-16` | $\sqrt{2}$ has no finite binary expansion | more bits only shrink it: `sqrt(big(2)) * sqrt(big(2)) - 2` → `-1.7272…e-77`; compare with `≈` (next table) |
| float rounding of π | `sin(1pi)` → `1.2246467991473532e-16` | `1pi` is a `Float64`, the float nearest π but not π (`1pi == pi` → `false`), so its sine is not 0 | keep `pi` an `Irrational`: `sin(pi)` → `0.0`, because Base defines `sin(::Irrational{:π}) = 0.0` (`mathconstants.jl:146`) |
| float overflow | `2.0^1024` → `Inf` | above `floatmax(Float64)` → `1.7976931348623157e308` | `big(2.0)^1024` → `1.7976931348623159…e308` |

`≈` is typed `\approx` then Tab; it is the same function as `isapprox(a, b)`.

| comparison | with `==` | with `≈` |
|---|---|---|
| `0.1 + 0.2` and `0.3` | `false` | `true` |
| `sqrt(2) * sqrt(2)` and `2` | `false` | `true` |
| `1/10 + (2/10 + 3/10)` and `(1/10 + 2/10) + 3/10` | `false` | `true` |
| `0.1 + 0.2 - 0.3` and `0` | `false` | `false`: the trap (below) |
| `isapprox(0.1 + 0.2 - 0.3, 0; atol=1e-12)` | not applicable | `true` |

The trap: by default `≈` uses a *relative* tolerance of about `1.49e-8` (`sqrt(eps(1.0))`), scaled by the larger of the two numbers. `isapprox(1.0, 1.0 + 1e-9)` → `true`, but `isapprox(1.0, 1.0 + 1e-7)` → `false`. Against `0` the tolerance scales down with the other number, so any non-zero value fails. `0.1 + 0.2 - 0.3` is `5.551115123125783e-17`, not `0`. To compare with zero, give an absolute tolerance with `atol`.

#### Abstract and concrete types

Every value in Julia has exactly one *concrete* type, which `typeof` returns: `typeof(1)` → `Int64`, `typeof(1.0)` → `Float64`. A concrete type exactly describes how the value is stored in computer memory. `Int8(1)` and `1` are the same integer, but `sizeof` gives 1 byte for the first and 8 for the second.

An *abstract* type is a name for a group of types. It allows you to specify what kind of value a variable or function argument might contain, without determining up front how it is stored in memory. No value has an abstract type as its `typeof`. You can only ask whether a value belongs to one, with `isa`: `1 isa Integer` → `true`, `1 isa Real` → `true`, `1.0 isa Integer` → `false`. 

Types form a hierarchy, a tree with `Any` at the top and the concrete types as its leaves. The function `supertypes` lists the chain upward from a concrete type:

- `supertypes(Int64)` → `(Int64, Signed, Integer, Real, Number, Any)`
- `supertypes(Float64)` → `(Float64, AbstractFloat, Real, Number, Any)`
- `supertypes(Complex{Int64})` → `(Complex{Int64}, Number, Any)`: `Complex` sits directly under `Number`, which is why a complex number is a `Number` but not a `Real`.

It's important to note: Verzani's remark that `Rational` and `Complex` "aren't even concrete types" does not mean they are abstract. They are *parametric* types: families that become concrete once the type of their parts is given. `typeof(1//1)` → `Rational{Int64}`, `typeof(1 + 0im)` → `Complex{Int64}`, `typeof(1.0 + 0im)` → `ComplexF64` (short for `Complex{Float64}`).

| type | kind | `isabstracttype` | `isconcretetype` |
|---|---|---|---|
| `Number`, `Real`, `Integer` | abstract | `true` | `false` |
| `Int64`, `Int8`, `BigInt`, `Float64` | concrete | `false` | `true` |
| `Rational`, `Complex` | parametric family | `false` | `false` |
| `Rational{Int64}`, `Complex{Float64}` | concrete | `false` | `true` |

Abstract types let one function cover many concrete types. With `twice(x::Real) = 2x`, `twice(3)` → `6`, `twice(1.5)` → `3.0`, `twice(1//3)` → `2//3` and `twice(big(10)^20)` → `200000000000000000000`. But `twice(1 + 2im)` → `MethodError: no method matching twice(::Complex{Int64})`, because `Complex` is not `Real`.

`supertypes` comes from the standard library `InteractiveUtils`, and is not always available in your workspace. The Julia REPL loads `InteractiveUtils` for you; `using Calculus` does not. Hence, if Julia says `supertypes` is not defined, run `using InteractiveUtils` first. By contrast, Base's `supertype` (singular) always works and gives one step at a time: `supertype(Int64)` → `Signed`.

#### Scientific notation: base 10 on Wikipedia, base 2 in a `Float64`

Verzani in this chapter writes a float as $a\cdot 2^b$ with $1 \leq \lvert a\rvert < 2$; the Wikipedia article he links uses $a\cdot 10^b$ with $1 \leq \lvert a\rvert < 10$. The idea is the same in any base $\beta$: $a\cdot\beta^b$ with $1 \leq \lvert a\rvert < \beta$. A `Float64` uses $\beta = 2$ because the hardware stores bits.

| `x` | `significand(x)` | `exponent(x)` | so `x` is |
|---|---|---|---|
| `6.0` | `1.5` | `2` | $1.5\cdot 2^{2}$ |
| `0.1` | `1.6` | `-4` | $1.6\cdot 2^{-4}$ (nearest float, not exact) |
| `1e-21` | `1.1805916207174112` | `-70` | $1.18\ldots\cdot 2^{-70}$ |
| `1e300` | `1.4932217896051503` | `996` | $1.49\ldots\cdot 2^{996}$ |

`1e-21` is base-10 notation only on entry and display: Julia converts it to the nearest base-2 float. That conversion is where $1/10$ loses exactness — it is finite in base 10 ($0.1$) but has no finite base-2 expansion: `big(0.1)` → `0.1000000000000000055511151231257827…`. Hence `0.1 + 0.2 == 0.3` → `false`, while `0.5 + 0.25 == 0.75` → `true` ($2^{-1}$ and $2^{-2}$ are exact in base 2).